> 在线讲义：[第 17 章 · Nav2：移动机器人导航栈](https://wikieden.github.io/ros2-from-scratch/lessons/17-nav2.html) · 本书《从零学 ROS 2》

# 第 17 章 · Nav2：移动机器人导航栈

本章回答：在 RViz 里点一个目标点，机器人是怎么一路避开障碍开过去的？Nav2 由哪些服务器组成，各管什么？代价地图是什么？行为树在里面起什么作用？卡住了怎么办？

配套讲义：仓库里 `ros2/notes/lessons/17-nav2.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 Nav2。这里用一张 5 × 9 的小格子图：'#' 是障碍（被占），膨胀层给障碍周围的格子加代价，规划器用 Dijkstra（NavFn 插件的算法之一）找总代价最小的路。膨胀代价的数是编的，只表示"离障碍越近越贵"，
不是 Nav2 膨胀层的真实公式。

In [ ]:
import heapq
import math

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 2 步 代价地图 | 被占 / 膨胀 / 空闲 | `inflate(GRID)` | 墙两侧和墙脚的格子标成 '+' |
| 第 1 步 规划器（无膨胀） | Dijkstra 找总代价最小 | `dijkstra(GRID, START, GOAL, {})` | 14 格，贴着墙脚绕 |
| 第 2 步 加膨胀层 | 离障碍越近越贵 | `dijkstra(..., extra)` | 16 格，路上一个 '+' 都不经过 |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**常量 `START`, `GOAL`**

In [ ]:
START, GOAL = (0, 0), (0, 8)

**常量 `GRID`**

In [ ]:
GRID = [
    "....#....",
    "....#....",
    "....#....",
    ".........",
    ".........",
]  # 一堵从上往下的墙；起点左上、终点右上（编的）

**函数 `obstacles`**：下面要用到的小工具。

In [ ]:
def obstacles(grid: list[str]) -> set[tuple[int, int]]:
    return {(r, c) for r, row in enumerate(grid) for c, ch in enumerate(row) if ch == "#"}

**函数 `dijkstra`**：上下左右走一格代价 1，再加上格子的膨胀代价；障碍格不能进。返回路径和总代价。

In [ ]:
def dijkstra(grid: list[str], start, goal, extra: dict) -> tuple[list[tuple[int, int]], float]:
    """上下左右走一格代价 1，再加上格子的膨胀代价；障碍格不能进。返回路径和总代价。"""
    obs = obstacles(grid)
    rows, cols = len(grid), len(grid[0])
    best = {start: 0.0}
    prev = {}
    todo = [(0.0, start)]
    while todo:
        d, cell = heapq.heappop(todo)
        if cell == goal:
            break
        if d > best[cell]:
            continue
        r, c = cell
        for nb in [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]:
            if not (0 <= nb[0] < rows and 0 <= nb[1] < cols) or nb in obs:
                continue
            nd = d + 1 + extra.get(nb, 0.0)
            if nd < best.get(nb, math.inf):
                best[nb], prev[nb] = nd, cell
                heapq.heappush(todo, (nd, nb))
    path = [goal]
    while path[-1] != start:
        path.append(prev[path[-1]])
    return path[::-1], best[goal]

**函数 `draw`**：'#' 障碍、'+' 膨胀过的格子、'*' 路径、S / G 起点终点。

In [ ]:
def draw(grid: list[str], path, extra: dict) -> list[str]:
    """'#' 障碍、'+' 膨胀过的格子、'*' 路径、S / G 起点终点。"""
    out = []
    on_path = set(path)
    for r, row in enumerate(grid):
        line = ""
        for c, ch in enumerate(row):
            if (r, c) == START:
                line += "S"
            elif (r, c) == GOAL:
                line += "G"
            elif (r, c) in on_path:
                line += "*"
            elif (r, c) in extra:
                line += "+"
            else:
                line += ch
        out.append(line)
    return out

**函数 `inflate`**：膨胀层：离最近障碍的距离 d ≤ radius 的格子加代价，越近越贵（d=1 加 3，d=√2 加 1.5）。

In [ ]:
def inflate(grid: list[str], radius: float = 1.5) -> dict[tuple[int, int], float]:
    """膨胀层：离最近障碍的距离 d ≤ radius 的格子加代价，越近越贵（d=1 加 3，d=√2 加 1.5）。"""
    obs = obstacles(grid)
    cost = {}
    for r in range(len(grid)):
        for c in range(len(grid[0])):
            if (r, c) in obs:
                continue
            d = min(math.dist((r, c), o) for o in obs)
            if d <= radius:
                cost[(r, c)] = 3.0 if d <= 1 else 1.5
    return cost

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 · 代价地图：'#' 被占，'+' 是膨胀层加了代价的格子")
extra = inflate(GRID)
for line in draw(GRID, [], extra):
    print("  ", line)

In [ ]:
print("\n第一步 · 规划器：没有膨胀层时，最短路贴着墙脚绕过去")
path, cost = dijkstra(GRID, START, GOAL, {})
for line in draw(GRID, path, {}):
    print("  ", line)
print(f"  走了 {len(path) - 1} 格，总代价 {cost}")

In [ ]:
print("\n第二步（续）· 加上膨胀层：多走两格，换来离墙远一点")
path, cost = dijkstra(GRID, START, GOAL, extra)
for line in draw(GRID, path, extra):
    print("  ", line)
print(f"  走了 {len(path) - 1} 格，总代价 {cost}")

## 3. 代码里才看得到的

1. **膨胀代价是编的**（距离 1 加 3、距离 √2 加 1.5），只表示"越近越贵"；Nav2 的膨胀层按指数衰减（`first_time_robot_setup_guide/sensors/mapping_localization.md` L126）。
2. **Dijkstra 和 A\***：NavFn 插件两种都支持。A\* 多加一项"离终点还有多远"的估计，展开的格子更少，路径代价一样。
3. **只能上下左右走**：真代价地图允许斜着走，路径会更顺。

## 4. 自己改一改

- 把 `GOAL` 改成 (4, 8)，看两种情况的路径；
- 在 `GRID` 第 4 行加一个 '#'，看规划器怎么绕。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **五服务器**：规划算路、控制跟路、平滑修路、行为救急、路线走图；算法都是插件，用别名调。
- **两张代价地图**：全局在 map 给规划，局部在 odom 给控制；静态层、障碍层、膨胀层叠起来。
- **行为树**：每秒重规划，失败先清图，再转、等、退，最多重试六次；新目标随时抢占。
- **前提**：tf 主干齐全，全员托管节点由 lifecycle_manager 拉起。